# 06b - SHAP Global Explanations + ICE Plots
Globalni vyznam promennych pro vsechny tri tuned modely.

- **Beeswarm plot** — jak kazda hodnota featury (vysoka/nizka) ovlivnuje predikci
- **Bar plot** — serazeni promennych podle prumerne absolutni SHAP hodnoty
- **Srovnavaci heatmapa** — ktery model povazuje ktery priznak za dulezity
- **ICE plots (Individual Conditional Expectation)** — jak se predikce meni pri zmene jedne featury pro kazdeho studenta zvlast; centrovane ICE (c-ICE) odecita bazovou hodnotu pro vizualizaci heterogenity efektu

**Explainer volime podle typu modelu:**
- `LinearExplainer` — pro Logistic Regression
- `TreeExplainer` — pro Random Forest a Gradient Boosting

In [1]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import joblib

ctx = joblib.load('../results/xai/xai_context.pkl')
X_train_prep     = ctx['X_train_prep']
X_test_prep      = ctx['X_test_prep']
X_test_prep      = X_test_prep.astype(float)
X_train_prep     = X_train_prep.astype(float)
tuned_models     = ctx['tuned_models']
student_index    = ctx['student_index']
INTEREST_FEATURE = ctx.get('INTEREST_FEATURE', 'Study_Hours_per_Day')

print(f"Atribut zájmu: {INTEREST_FEATURE}, student_index: {student_index}")

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


Atribut zájmu: Study_Hours_per_Day, student_index: 17


## Pomocná funkce: výpočet SHAP hodnot
Abstrahujeme volbu explaineru podle typu modelu, aby byl kód DRY.

In [2]:
def get_shap_values(model, X_train, X_test):
    model_type = type(model).__name__
    if model_type == 'LogisticRegression':
        explainer = shap.LinearExplainer(model, X_train)
        shap_vals = explainer(X_test)
        shap_vals.values = shap_vals.values.astype(float)
        shap_vals.data   = np.array(shap_vals.data, dtype=float)
    else:
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer(X_test)

    # Pokud SHAP vrátil 3D pole (n_samples, n_features, n_classes),
    # vezmeme jen třídu 1 (Dropout) — index -1 funguje pro oba případy (2 i více tříd)
    if shap_vals.values.ndim == 3:
        shap_vals.values = shap_vals.values[:, :, 1]
        if hasattr(shap_vals, 'base_values') and np.ndim(shap_vals.base_values) == 2:
            shap_vals.base_values = shap_vals.base_values[:, 1]

    return shap_vals

print("Pocitam SHAP hodnoty pro vsechny modely (moze chvili trvat)...")
shap_values_all = {}
for name, model in tuned_models.items():
    print(f"  {name}...")
    shap_values_all[name] = get_shap_values(model, X_train_prep, X_test_prep)
    print(f"    shape: {shap_values_all[name].values.shape}")
print("Hotovo.")

Pocitam SHAP hodnoty pro vsechny modely (moze chvili trvat)...
  Logistic Regression...
    shape: (2000, 19)
  Random Forest...


    shape: (2000, 19)
  Gradient Boosting...
    shape: (2000, 19)
  Decision Tree...
    shape: (2000, 19)
Hotovo.


## Beeswarm plot — jak featury ovlivňují predikci
Kazdý bod = jeden student. Barva = hodnota featury (cervena = vysoka, modra = nizka). Pozice na ose X = vliv na predikci dropoutu.

In [3]:
for name, shap_vals in shap_values_all.items():
    print(f"\n{'=' * 55}")
    print(f"  SHAP Beeswarm: {name}")
    print(f"{'=' * 55}")

    # Dynamically handle different SHAP dimensions
    if len(shap_vals.shape) == 3:
        # If the 3rd dimension has 2 classes, grab class 1 (usually the positive class)
        # If it only has 1 class, grab class 0
        class_index = 1 if shap_vals.shape[2] > 1 else 0
        vals_to_plot = shap_vals[:, :, class_index]
    else:
        # If it's already 2D (samples, features), no slicing is needed
        vals_to_plot = shap_vals

    # Plot the safely sliced values
    shap.plots.beeswarm(vals_to_plot, max_display=15, show=True)


  SHAP Beeswarm: Logistic Regression



  SHAP Beeswarm: Random Forest



  SHAP Beeswarm: Gradient Boosting



  SHAP Beeswarm: Decision Tree


## Bar plot — průměrná absolutní důležitost
Agregovanejsi pohled — vhodny pro rychle porovnani.

In [4]:
for name, shap_vals in shap_values_all.items():
    print(f"\n{'=' * 55}")
    print(f"  SHAP Bar: {name}")
    print(f"{'=' * 55}")

    # Dynamically slice to 2D, just like the beeswarm fix
    if len(shap_vals.shape) == 3:
        class_index = 1 if shap_vals.shape[2] > 1 else 0
        vals_to_plot = shap_vals[:, :, class_index]
    else:
        vals_to_plot = shap_vals

    # Plot the safely sliced values
    shap.plots.bar(vals_to_plot, max_display=15, show=True)


  SHAP Bar: Logistic Regression

  SHAP Bar: Random Forest



  SHAP Bar: Gradient Boosting

  SHAP Bar: Decision Tree


## Srovnávací přehled: top featury napříč modely
Ktery priznak se objevi v top-10 u vsech tri modelu?

In [5]:
import pandas as pd
import matplotlib.pyplot as plt

top_n = 10
summary_rows = []

for name, shap_vals in shap_values_all.items():
    vals = shap_vals.values
    # Normalizace: TreeExplainer muze vratit (n_samples, n_features, n_classes)
    # Bereme tridu 1 (Dropout); pro 2D pole nedelame nic
    if vals.ndim == 3:
        vals = vals[:, :, 1]
    mean_abs = np.abs(vals).mean(axis=0)
    feature_importance = pd.Series(mean_abs, index=X_test_prep.columns)
    top_features = feature_importance.nlargest(top_n)
    for rank, (feat, imp) in enumerate(top_features.items(), 1):
        summary_rows.append({'Model': name, 'Feature': feat, 'Rank': rank, 'Mean |SHAP|': imp})

df_summary = pd.DataFrame(summary_rows)

pivot = df_summary.pivot_table(index='Feature', columns='Model', values='Mean |SHAP|', aggfunc='first').fillna(0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 7))
import seaborn as sns
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean |SHAP|'}
)
ax.set_title('Prumerna absolutni SHAP hodnota: porovnani modelu', fontsize=13, pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## ICE Plots — Individual Conditional Expectation
Kazda cara = jeden student. Ukazuje, jak by se jeho predikce zmenila, kdyby se zmenila pouze jedna featura — zbytek zustava stejny.

**Proc ICE a ne jen PDP (Partial Dependence Plot)?**
PDP ukazuje prumer pres vsechny studenty — moze maskovat heterogenitu. ICE ukazuje, ze nekteri studenti reagují na zmenu featury opacne nez ostatni (efekt interakci). Centrovane ICE (c-ICE) normalizuje vsechny cary na spolecny zacatek (0), aby byl smer efektu citatejlny bez vlivu absolutni urovne.

**Featury pro ICE** volime na zaklade SHAP konsensu z predchozi sekce — top-4 nejdulezitejsi.

In [6]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import numpy as np

# Top featury podle SHAP konsensu — iterujeme pres X_test_prep.columns (zarucene spravne nazvy)
shap_consensus = {}
for shap_vals in shap_values_all.values():
    vals = shap_vals.values
    if vals.ndim == 3:
        vals = vals[:, :, 1]
    mean_abs = np.abs(vals).mean(axis=0)
    for i, feat in enumerate(X_test_prep.columns):
        shap_consensus[feat] = shap_consensus.get(feat, 0) + mean_abs[i]

top_ice_features = sorted(X_test_prep.columns, key=lambda f: shap_consensus[f], reverse=True)[:4]
print(f"ICE featury (SHAP konsensus top-4): {top_ice_features}")

col_list = list(X_test_prep.columns)

for name, model in tuned_models.items():
    print(f"\n{'=' * 55}")
    print(f"  ICE + PDP: {name}")
    print(f"{'=' * 55}")

    feat_indices = [col_list.index(f) for f in top_ice_features]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for ax, feat_idx, feat_name in zip(axes, feat_indices, top_ice_features):
        PartialDependenceDisplay.from_estimator(
            model,
            X_test_prep,
            features=[feat_idx],
            kind='both',
            subsample=200,
            random_state=42,
            ax=ax,
            ice_lines_kw={'alpha': 0.08, 'color': 'steelblue', 'linewidth': 0.8},
            pd_line_kw={'color': '#D85A30', 'linewidth': 2.5, 'label': 'PDP (prumer)'},
            feature_names=col_list
        )
        ax.set_title(feat_name, fontsize=11)
        ax.set_xlabel(feat_name)
        ax.set_ylabel('Predikce (pravdepodobnost dropoutu)')

    fig.suptitle(f'ICE Plots: {name}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

ICE featury (SHAP konsensus top-4): ['CGPA', 'Stress_Index', 'Scholarship', 'Attendance_Rate']

  ICE + PDP: Logistic Regression



  ICE + PDP: Random Forest



  ICE + PDP: Gradient Boosting



  ICE + PDP: Decision Tree


## Centrované ICE (c-ICE) — heterogenita efektů
c-ICE odecte od kazde cary jeji hodnotu v levem krajnim bode (minimum featury). Vsechny cary zacínají na 0 — vidíme pouze *smer a intenzitu zmeny*, ne absolutni uroven.

Pokud se cary rozbíhají (diverguji), featura ma heterogenni efekt — u nekterych studentu zvysuje riziko vice nez u jinych. Pokud jsou cary paralélní, efekt je homogenní (bez interakcí).

In [7]:
from sklearn.inspection import PartialDependenceDisplay

col_list = list(X_test_prep.columns)

# Pojistka: filtrujeme top_ice_features na featury, ktere skutecne existuji v X_test_prep
# (ochrana pred stale stavem kernelu pokud byla predchozi bunka zmenena bez restartu)
valid_ice_features = [f for f in top_ice_features if f in col_list]
if len(valid_ice_features) < len(top_ice_features):
    chybejici = [f for f in top_ice_features if f not in col_list]
    print(f"VAROVANI: nasledujici featury nejsou v X_test_prep a budou preskoceny: {chybejici}")
if len(valid_ice_features) == 0:
    raise ValueError("Zadna z top_ice_features neni v X_test_prep.columns — spust znovu bunku s vypoctem top_ice_features.")

feat_indices = [col_list.index(f) for f in valid_ice_features]

for name, model in tuned_models.items():
    print(f"\n{'=' * 55}")
    print(f"  Centrovane ICE (c-ICE): {name}")
    print(f"{'=' * 55}")

    n_feats = len(valid_ice_features)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for ax, feat_idx, feat_name in zip(axes, feat_indices, valid_ice_features):
        PartialDependenceDisplay.from_estimator(
            model,
            X_test_prep,
            features=[feat_idx],
            kind='individual',
            centered=True,
            subsample=200,
            random_state=42,
            ax=ax,
            ice_lines_kw={'alpha': 0.1, 'color': 'steelblue', 'linewidth': 0.8},
            feature_names=col_list
        )
        ax.set_title(f'{feat_name} (c-ICE)', fontsize=11)
        ax.set_xlabel(feat_name)
        ax.set_ylabel('Delta predikce od bazove hodnoty')
        ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')

    # Skryjeme prazdne subploty pokud je valid_ice_features < 4
    for i in range(n_feats, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle(f'Centrovane ICE (c-ICE): {name}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


  Centrovane ICE (c-ICE): Logistic Regression



  Centrovane ICE (c-ICE): Random Forest



  Centrovane ICE (c-ICE): Gradient Boosting



  Centrovane ICE (c-ICE): Decision Tree


/Users/janvrzal/Documents/Uni/Zpracování informací a znalostí/student dropout/.venv/lib/python3.12/site-packages/sklearn/inspection/_plot/partial_dependence.py:990: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim([min_val, max_val])


## Uložení SHAP hodnot

## ICE pro atribut zájmu — vybraná instance
Cílený ICE plot pro `INTEREST_FEATURE` (Study_Hours_per_Day) z kontextu 06a. Červená svislá čára ukazuje skutečnou hodnotu vybraného studenta. Vidíme, jak by se jeho predikce změnila, kdyby studoval více/méně hodin.

Tento plot přímé odpovídá otázce zadání: *"Pokud bychom změnili atribut zájmu, zlepšila by se predikce?"*

In [8]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import numpy as np
import joblib

ctx             = joblib.load('../results/xai/xai_context.pkl')
INTEREST_FEATURE = ctx['INTEREST_FEATURE']
student_index   = ctx['student_index']
X_test_full     = ctx['X_test_prep'].astype(float)

col_list = list(X_test_full.columns)
feat_idx = col_list.index(INTEREST_FEATURE)
student_val = float(X_test_full.iloc[student_index][INTEREST_FEATURE])

print(f"Atribut zajmu : {INTEREST_FEATURE}")
print(f"Index studenta: {student_index}")
print(f"Hodnota u studenta: {student_val:.4f}")

n_models = len(tuned_models)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=False)
if n_models == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, tuned_models.items()):
    PartialDependenceDisplay.from_estimator(
        model,
        X_test_full,
        features=[feat_idx],
        kind='both',
        subsample=200,
        centered=False,
        random_state=42,
        ax=ax,
        ice_lines_kw={'alpha': 0.08, 'color': 'steelblue', 'linewidth': 0.8},
        pd_line_kw={'color': '#D85A30', 'linewidth': 2.5, 'label': 'PDP (průměr)'},
        feature_names=col_list
    )
    # Zvýraznit vybraného studenta
    ax.axvline(
        student_val, color='crimson', linestyle='--', linewidth=2,
        label=f'Vybraný student (={student_val:.2f})'
    )
    ax.set_title(f'{name}', fontsize=11)
    ax.set_xlabel(INTEREST_FEATURE)
    ax.set_ylabel('Predikce (pravděpodobnost dropoutu)')
    ax.legend(fontsize=8)

fig.suptitle(
    f'ICE / PDP pro atribut zájmu: {INTEREST_FEATURE}\n'
    f'(červená čára = hodnota vybraného studenta)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(f'../results/xai/ice_interest_feature.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Graf ulozen: results/xai/ice_interest_feature.png")


Atribut zajmu : Study_Hours_per_Day
Index studenta: 17
Hodnota u studenta: -2.1000


Graf ulozen: results/xai/ice_interest_feature.png


In [9]:
import joblib
joblib.dump(shap_values_all, '../results/xai/shap_values_all.pkl')
print("SHAP hodnoty ulozeny do results/xai/shap_values_all.pkl")

SHAP hodnoty ulozeny do results/xai/shap_values_all.pkl
